In [2]:
import gc
import os
import sys
import logging
import warnings
from pathlib import Path
import matplotlib as mpl
import pickle
import session_info

HOMEDIR = Path('../../../../..')
SRCDIR = HOMEDIR / 'src'
PLOTDIR = HOMEDIR / "plots" / "kennedi_xenium"
DATADIR = HOMEDIR / "data" / "processed" / "spatial" / "Xenium" / "kennedi_flu"
if not str(SRCDIR) in sys.path:
    sys.path.insert(0, str(SRCDIR))

logging.basicConfig(level="WARNING")
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", DeprecationWarning)

from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from spatial_seq.plot import *
from utils import *

import squidpy as sq
import cellcharter as cc
# import spatialdata as sd
# import spatialdata_plot as sdp

CORES = 20
%matplotlib inline
# R_preload()
mpl.rcdefaults()
plt.rcParams["figure.figsize"] = (8, 8)
gc.collect()

session_info.show()

/mnt/DATA/home/ethung/projects/spatial_seq/.pixi/envs/main/lib/python3.12/site-packages/phenograph/cluster.py:13: DeprecationWarning: Please import `spmatrix` from the `scipy.sparse` namespace; the `scipy.sparse.base` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.sparse.base import spmatrix
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
/mnt/DATA/home/ethung/projects/spatial_seq/.pixi/envs/main/lib/python3.12/site-packages/ome_zarr/writer.py:967: DeprecationWarning: Call to deprecated class Scaler. (Downsampling via the `Scaler` class has been deprecated. Please use the `scale_factors` argument instead.) -- Deprecated since version 0.14.0.
/mnt/DATA/home/ethung/projects/spatial_seq/.pi

In [36]:
adata = sc.read_h5ad(DATADIR / "integrated.h5ad")
adata

AnnData object with n_obs × n_vars = 4654531 × 480
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'Slide', 'Sample', 'n_genes', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'Genotype', 'Timepoint', 'cellcharter_k3', 'cellcharter_k6', 'cellcharter_k12', 'LabelTransfer_OT', 'Groups'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mean', 'std'
    uns: 'LabelTransfer_OT_nhood_enrichment', 'PCA', 'UMAP', 'cellcharter_k12_LabelTransfer_OT_enrichment', 'cellcharter_k12_nhood_enrichment', 'cellcharter_k6_LabelTransfer_OT_enrichment', 'cellcharter_k6_nhood_enrichment', 'dendrogram_cellchar

In [37]:
df = pd.read_csv(DATADIR / "integrated_metadata.csv", index_col="cell_id")
adata = adata[adata.obs_names.isin(df.index)]
adata.obs["celltype0518"] = df["celltype0518"]

/tmp/ipykernel_230181/646104679.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


In [38]:
adata.write(DATADIR / "integrated.h5ad")
adata

AnnData object with n_obs × n_vars = 4522991 × 480
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'Slide', 'Sample', 'n_genes', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'Genotype', 'Timepoint', 'cellcharter_k3', 'cellcharter_k6', 'cellcharter_k12', 'LabelTransfer_OT', 'Groups', 'celltype0518'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mean', 'std'
    uns: 'LabelTransfer_OT_nhood_enrichment', 'PCA', 'UMAP', 'cellcharter_k12_LabelTransfer_OT_enrichment', 'cellcharter_k12_nhood_enrichment', 'cellcharter_k6_LabelTransfer_OT_enrichment', 'cellcharter_k6_nhood_enrichment', 'den

# SLIM (for RDS conversion)

In [39]:
del adata.uns, adata.varm, adata.obsp, adata.obsm["HotspotModule"], adata.obsm["LT_OT"], adata.obsm["global_leiden"]

In [40]:
adata.write(DATADIR / "integrated_slim.h5ad")
adata

AnnData object with n_obs × n_vars = 4522991 × 480
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'Slide', 'Sample', 'n_genes', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'Genotype', 'Timepoint', 'cellcharter_k3', 'cellcharter_k6', 'cellcharter_k12', 'LabelTransfer_OT', 'Groups', 'celltype0518'
    var: 'n_cells', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mean', 'std'
    obsm: 'PCA', 'SPATIAL', 'UMAP', 'integrated', 'spatial'
    layers: 'counts', 'normalized'